### Middlewares
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [7]:
# Import load_dotenv to load API keys and environment variables from the .env file
from dotenv import load_dotenv

# Import create_agent to create an AI agent
from langchain.agents import create_agent

# Import SummarizationMiddleware to summarize older messages when the conversation becomes long
from langchain.agents.middleware import SummarizationMiddleware

# Import InMemorySaver to save conversation history temporarily in memory
from langgraph.checkpoint.memory import InMemorySaver

# Import HumanMessage to represent messages sent by the user
from langchain_core.messages import HumanMessage

# Import ChatGoogleGenerativeAI to connect LangChain with Google's Gemini models
from langchain_google_genai import ChatGoogleGenerativeAI


# Load the API key from the .env file so that Gemini can access it
load_dotenv()


# Initialize the Gemini model that will act as the brain of our AI agent
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # Specify which Gemini model we want to use
    temperature=0  # Set temperature to 0 to get more consistent and less random answers
)


# Create an AI agent by connecting the Gemini model with memory and summarization
agent = create_agent(
    model=llm,  # Tell the agent to use our initialized Gemini model

    checkpointer=InMemorySaver(),  # Save previous conversation states so the agent can remember earlier messages

    middleware=[
        SummarizationMiddleware(
            model=llm,  # Use Gemini to summarize older conversation messages
            trigger=("messages", 10),  # Start summarization when the configured message-count threshold reaches 10
            keep=("messages", 4)  # Keep the latest 4 messages and summarize older conversation history
        )
    ]
)


# Create a configuration dictionary to identify and maintain a particular conversation
config = {
    "configurable": {
        "thread_id": "test_1"  # Give this conversation a unique ID so previous messages can be retrieved
    }
}


# Create a list containing multiple questions that we want to ask the AI agent
questions = [
    "What is 2*2",
    "What is 7/3",
    "What is 3+4",
    "What is 12/4",
    "What is 2+3",
    "What is 9+6",
]


# Use a for loop to take each question from the list one by one
for q in questions:

    # Send the current question to the AI agent and store its response
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},  # Convert the current question into a HumanMessage and send it to the agent
        config=config  # Use the same thread ID so that the agent continues the existing conversation
    )

    # Print the content of the latest message, which contains the agent's latest answer
    print(f"Response: {response['messages'][-1].content}")

    # Count and print the total number of messages currently present in the returned conversation state
    print(f"Total Messages: {len(response['messages'])}")

    # Print a separator line to make the output of each question easier to read
    print("-" * 50)

Response: [{'type': 'text', 'text': '2 * 2 = 4', 'extras': {'signature': 'EqEBCp4BAWkUfRPayUizVJCJYq9BxE5Fih48oeHByG2ARuPAeogzcA7LVjB5dFsB4TJ75kC7YmVx7cas6ayR7zEmB6vjnqnm82r6eqlMwCXMxTl8JzzYm83atqEsKtvTcj15aZQadgf9nv4f8EIEVPUgyPhXR69adtYqD+DYFnMG9q/OZlZf1HDTttIsX51aq2KEWUMTz1ZibC3+RRqUrDTIx4U='}}]
Total Messages: 2
--------------------------------------------------
Response: [{'type': 'text', 'text': '7 / 3 is approximately **2.33** (or 2 and 1/3).', 'extras': {'signature': 'EqEBCp4BAWkUfRO52jcLZYQUfZQMRtvZtS3aTDmGdkNG/ZGwvtkGusslQc8s+Alafm6ZolYZDY23jMIn2pWLIhYryQKTYV+mxtWycKHkEdjyflgBuH4ZgTMypOUcwgoDztIleYJXmxFUtJNfiCOYPQcmPyEwDXnw0RvvdyxRPEKH7y2tSaol13wzy5TfuMAPmJUFDBrD6vBL5Xsd0ofptGIcRxU='}}]
Total Messages: 4
--------------------------------------------------
Response: [{'type': 'text', 'text': '3 + 4 = 7', 'extras': {'signature': 'EqEBCp4BAWkUfROWm7qx7AlDEuXDmTXL9ajxtrYsBTL6B24OX78rOQmXKhnav5S/XXgixBojCc33HRL/jEeN6yTouvC3bkbNSdclnwWi79bKPsCec7Hovs8Nj8iZpkHY58uSYU/CvZAyjj04JKRk8zuS

Above was based on msg size, next is based on token size
### MiddleWARE on TOKEN SIZE


In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # Specify which Gemini model we want to use
    temperature=0  # Set temperature to 0 to get more consistent and less random answers
)
agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),#if it hits 550 tokens, then
            keep=("tokens",200)#keep the memory of last 200 tokens
        )
    ]

)
config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


In [12]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~179 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='61e5842d-bb54-4c1b-a0be-18e4fe8d4499'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_16633635353571733452': 'EqEBCp4BAWkUfRM/RzzO1kWNRbHUGF3vdi/+UqqGwkd4cpOcw5GYEr1JNxiDMuYy/FV7D+N0bBMK6JGZ1eAlAlpoMG1gedXrGyoDDTpDPWXHaTMssq0/XsQDVy2jaOJAKB83ZuneSnlNshuba/GbDCwosiAB3zjuMtl38bReNy5o+sc+6efurl8OmM1WdhejxFqG2A0wfavnEkiMuWQMaayv4+M='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'}, id='lc_run--01a2045a-5a59-747a-afc2-0d794323f34c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_16633635353571733452', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5

KeyboardInterrupt: 

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

In [15]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
# Create an AI agent with email tools and human supervision
agent = create_agent(
    model=llm,  # Use the LLM as the brain of the agent

    tools=[read_email_tool, send_email_tool],  # Give the agent tools to read and send emails

    checkpointer=InMemorySaver(),  # Save the agent's state so it can resume after human approval

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    # Ask for human permission before sending an email
                    "allowed_decisions": ["approve", "edit", "reject"]
                    # Approve = Send the email
                    # Edit = Modify the email before sending
                    # Reject = Cancel the email
                },

                "read_email_tool": False  # Allow reading emails without human approval
            }
        )
    ]
)

In [21]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fa157893-e082-49ed-911e-0e4c890bf3e0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_7935878364545693301': 'EqEBCp4BAWkUfRPLauM3ilCQn7NEKVAh6ZopiCp5G7rYhocTOf1bR9BDTSfsDD7Q/eG0a68Mxnr1QYTmCNnY0+DhVehaCfc/mq6rudnynLstz86HIB3bSDZpO/DkUN9OqDEZKpDSZ1gYW1BTmshvfos26mAJ5phIAC421ocJ6l99/QnUHlsWUTZBmgtEvkrmmH9iklA//569bV4NeERcgEaY3Es='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'}, id='lc_run--01a20467-5047-72a4-9e48-9371f447d67a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'jo

In [23]:
# Step 2: Approve the pending email action

if "__interrupt__" in result:  # Check whether the agent has paused and is waiting for human approval

    print("⏸️ Paused! Approving...")  # Inform the user that the agent is paused and approval is being given

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}  # Approve the pending action so the agent can continue
                ]
            }
        ),
        config=config  # Use the same conversation thread to resume the paused agent
    )

    # Print the final result after the approval process is completed
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EqEBCp4BAWkUfROIvLTc+dwT04Q54o8gcXonU4LksTpy9ij8+s4fS01n4mbvqYDRCJ0e3/5zYNQAD3KLdz5r6LND0kifqqKOVcbryxZZ0YTvep2ecqIPS9VxFopyNglsodaqTzmYX0Uv9fceGoZXWY6+/uLvOUAN9wSiSPKsY6hi5xG/DPnapa1mSTTenGHwBvYvi6tqV49CoKFaDgyhSLxaLnA='}}]
